# 02 — Tiền xử lý Visual

Đọc dữ liệu chung, kiểm tra một clip pilot và chuẩn bị môi trường OpenFace.
Notebook gồm 6 cell code (Task 2.1–2.6), mỗi cell có phần giải thích ngay phía trên.
Đã đối chiếu cấu trúc với 3 CSV: `manifest.csv`, `pilot_manifest.csv`, `metadata_issues.csv`.

## Task 2.1 — Thư viện và Google Drive
Import thư viện chung và kết nối Google Drive. Chạy notebook trên Google Colab.


In [ ]:
# code
from pathlib import Path, PurePosixPath
from fractions import Fraction
import json
import os
import platform
import shutil
import subprocess
import zipfile
import mimetypes
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from google.colab import drive
from IPython.display import display, Video

drive.mount('/content/drive')
print('Python:', platform.python_version())
print('NumPy:', np.__version__, '| pandas:', pd.__version__)


## Task 2.2 — Đường dẫn dữ liệu chung và báo cáo Visual
Đặt 3 file CSV nhóm cung cấp trong `/content/drive/MyDrive/CMU-MOSI/preprocessing/shared/`:
`manifest.csv`, `pilot_manifest.csv` và `metadata_issues.csv`.
File video `Raw.zip` đặt trong `/content/drive/MyDrive/CMU-MOSI/`.

Báo cáo Visual được lưu trên Drive; video trích vào `/content` là file tạm.
Chỉ sửa đường dẫn tại cell này nếu Drive của bạn có cấu trúc khác.


In [ ]:
# code
DATA_ROOT = Path('/content/drive/MyDrive/CMU-MOSI')
SHARED_ROOT = DATA_ROOT / 'preprocessing' / 'shared'
MANIFEST_PATH = SHARED_ROOT / 'manifest.csv'
PILOT_MANIFEST_PATH = SHARED_ROOT / 'pilot_manifest.csv'
METADATA_ISSUES_PATH = SHARED_ROOT / 'metadata_issues.csv'

VISUAL_ROOT = DATA_ROOT / 'preprocessing' / 'visual'
REPORT_DIR = VISUAL_ROOT / 'reports'
LOCAL_MEDIA_ROOT = Path('/content/mosi_visual/media')

# None: dùng ID đầu tiên trong pilot chung.
# Nếu nhóm đã thống nhất ID tích hợp khác, điền chính xác ID đó.
PILOT_ID_OVERRIDE = None

# Điền đường dẫn FeatureExtraction nếu bạn đã cài OpenFace ở vị trí riêng.
OPENFACE_BIN_OVERRIDE = None
OPENFACE_ROOT = Path('/content/OpenFace')

shared_paths = [MANIFEST_PATH, PILOT_MANIFEST_PATH, METADATA_ISSUES_PATH]
missing_paths = [str(path) for path in shared_paths if not path.is_file()]
if missing_paths:
    raise FileNotFoundError(
        'Thiếu file chung trên Google Drive:\n'
        + '\n'.join(missing_paths)
        + '\nĐặt 3 CSV nhóm cung cấp vào SHARED_ROOT hoặc sửa đường dẫn ở cell này.'
    )

for folder in [REPORT_DIR, LOCAL_MEDIA_ROOT]:
    folder.mkdir(parents=True, exist_ok=True)

display(pd.DataFrame({
    'File': [path.name for path in shared_paths],
    'Đường dẫn': [str(path) for path in shared_paths],
}))
print('ZIP mặc định theo manifest đã gửi:', DATA_ROOT / 'Raw.zip')
print('Báo cáo Visual:', REPORT_DIR)


## Task 2.3 — Kiểm tra manifest, pilot và cảnh báo metadata
Manifest quản lý ID, transcript, nhãn, split và video. Pilot là danh sách clip nhóm chọn chạy thử.
Kiểm tra ID, nhãn, split, kích thước video và đối chiếu từng dòng pilot với manifest.

Ba CSV đã gửi có **2.199 mẫu: 1.284 train, 229 valid, 686 test**;
pilot gồm **10 clip train**. Đây là số liệu từ file nhóm, không phải notebook tự chia hoặc bốc mẫu.
`metadata_issues.csv` ghi **4 mẫu trùng transcript và 1 mẫu lệch annotation**.
Đây là cảnh báo metadata, chưa phải lỗi phát hiện khuôn mặt. Giữ nguyên các mẫu và nhãn hồi quy.

Kết quả cần có: các kiểm tra đầu vào bằng 0, pilot khớp manifest,
bảng số mẫu từng split và danh sách cảnh báo được lưu trong báo cáo Visual.


In [ ]:
# code
# keep_default_na=False giữ nguyên các chuỗi như "NA", không tự biến chúng thành NaN.
id_types = {'id': 'string', 'video_id': 'string', 'clip_id': 'string'}
manifest = pd.read_csv(MANIFEST_PATH, dtype=id_types, keep_default_na=False)
pilot_source = pd.read_csv(PILOT_MANIFEST_PATH, dtype=id_types, keep_default_na=False)
metadata_issues = pd.read_csv(
    METADATA_ISSUES_PATH, dtype='string', keep_default_na=False,
)

required_columns = [
    'id', 'video_id', 'clip_id', 'split', 'raw_text', 'label',
    'source_archive', 'video_relpath', 'video_size_bytes',
]
missing_columns = [c for c in required_columns if c not in manifest.columns]
if missing_columns:
    raise ValueError(f'Manifest thiếu cột: {missing_columns}')
if manifest.empty:
    raise ValueError('Manifest không có mẫu.')
if 'id' not in pilot_source.columns or pilot_source.empty:
    raise ValueError('pilot_manifest.csv phải có cột id và ít nhất một mẫu.')
issue_columns = ['id', 'issue', 'detail']
missing_issue_columns = [c for c in issue_columns if c not in metadata_issues.columns]
if missing_issue_columns:
    raise ValueError(f'metadata_issues.csv thiếu cột: {missing_issue_columns}')

string_columns = [
    'id', 'video_id', 'clip_id', 'split', 'raw_text',
    'source_archive', 'video_relpath',
]
blank_rows = pd.DataFrame({
    c: manifest[c].astype('string').fillna('').str.strip().eq('')
    for c in string_columns
}).any(axis=1)
labels_numeric = pd.to_numeric(manifest['label'], errors='coerce')
sizes_numeric = pd.to_numeric(manifest['video_size_bytes'], errors='coerce')
expected_ids = manifest['video_id'] + '$_$' + manifest['clip_id']

input_checks = pd.Series({
    'Mẫu thiếu thông tin bắt buộc': int(blank_rows.sum()),
    'Mẫu thuộc nhóm trùng ID': int(manifest['id'].duplicated(keep=False).sum()),
    'Mẫu thuộc nhóm trùng video_id–clip_id': int(
        manifest.duplicated(['video_id', 'clip_id'], keep=False).sum()
    ),
    'ID ghép sai quy ước': int(manifest['id'].ne(expected_ids).fillna(True).sum()),
    'Nhãn không hợp lệ': int((
        ~np.isfinite(labels_numeric) | ~labels_numeric.between(-3, 3)
    ).sum()),
    'Split không hợp lệ': int((~manifest['split'].isin(['train', 'valid', 'test'])).sum()),
    'Video nguồn xuất hiện ở nhiều split': int(
        manifest.groupby('video_id')['split'].nunique().gt(1).sum()
    ),
    'Kích thước video không hợp lệ': int((
        ~np.isfinite(sizes_numeric) | sizes_numeric.le(0) | sizes_numeric.mod(1).ne(0)
    ).sum()),
}, name='Số lượng')

display(input_checks.to_frame())
input_checks.to_csv(REPORT_DIR / 'visual_manifest_checks.csv', encoding='utf-8')
if input_checks.gt(0).any():
    raise ValueError('Manifest chưa đạt kiểm tra. Đối chiếu với người tổng hợp.')

pilot_ids = pilot_source['id']
if pilot_ids.str.strip().eq('').any() or pilot_ids.duplicated().any():
    raise ValueError('Pilot có ID rỗng hoặc trùng.')
unknown_ids = pilot_ids[~pilot_ids.isin(manifest['id'])]
if not unknown_ids.empty:
    raise ValueError(f'Pilot có ID ngoài manifest: {unknown_ids.tolist()}')

# Manifest là nguồn chính; giữ nguyên ID và thứ tự trong pilot nhóm cung cấp.
pilot = manifest.set_index('id').loc[pilot_ids.tolist()].reset_index()
for column in required_columns:
    if column == 'id' or column not in pilot_source.columns:
        continue
    left = pilot_source[column].reset_index(drop=True)
    right = pilot[column].reset_index(drop=True)
    if column in ['label', 'video_size_bytes']:
        equal = pd.to_numeric(left, errors='coerce').eq(pd.to_numeric(right, errors='coerce'))
    else:
        equal = left.astype('string').eq(right.astype('string'))
    if not equal.fillna(False).all():
        raise ValueError(f'Pilot và manifest khác dữ liệu tại cột {column}.')

issue_blanks = metadata_issues[issue_columns].apply(
    lambda column: column.str.strip().eq('')
).any(axis=1)
unknown_issue_ids = metadata_issues.loc[
    ~metadata_issues['id'].isin(manifest['id']), 'id'
]
if issue_blanks.any() or not unknown_issue_ids.empty:
    raise ValueError('Cảnh báo metadata có thông tin rỗng hoặc ID ngoài manifest.')

split_summary = (
    manifest.groupby('split')
    .agg(so_mau=('id', 'size'), so_video_nguon=('video_id', 'nunique'))
    .reindex(['train', 'valid', 'test'], fill_value=0)
)
# Số đối chiếu từ CSV nhóm đã gửi; không dùng để sửa hoặc chia lại dữ liệu.
reference_counts = pd.Series({'train': 1284, 'valid': 229, 'test': 686})
split_summary['so_mau_tham_chieu'] = reference_counts
split_summary['khop_so_mau'] = split_summary['so_mau'].eq(reference_counts)

print('Tổng số mẫu:', len(manifest))
display(split_summary)
if not split_summary['khop_so_mau'].all():
    print('Số mẫu khác bản CSV đã gửi. Cần xác nhận phiên bản với người tổng hợp.')

print('Pilot chung:', len(pilot), 'clip')
display(pilot[['id', 'split', 'video_relpath']])
print('Cảnh báo metadata — giữ nguyên mẫu và nhãn:')
display(metadata_issues)
issue_summary = (
    metadata_issues.groupby('issue')
    .agg(so_dong_canh_bao=('id', 'size'), so_id=('id', 'nunique'))
)
display(issue_summary)

split_summary.to_csv(REPORT_DIR / 'visual_split_summary.csv', encoding='utf-8')
metadata_issues.to_csv(
    REPORT_DIR / 'visual_metadata_issues.csv', index=False, encoding='utf-8',
)
print('Các CSV dùng chung được giữ nguyên.')


## Task 2.4 — Xác định một clip pilot và file ZIP
Dùng clip đầu tiên trong pilot chung: **`7JsX8y1ysxY$_$12`** của bản CSV đã gửi.
Đây là điểm bắt đầu để tích hợp cùng nhóm, không phải clip do notebook bốc ngẫu nhiên.
Nếu nhóm đã thống nhất ID khác, điền `PILOT_ID_OVERRIDE` tại Task 2.2.

`source_archive` trong CSV là `Raw.zip`; `video_relpath` là đường dẫn bên trong ZIP.
Kiểm tra video xuất hiện đúng một lần và kích thước khớp manifest.
Kết quả cần có: ID pilot và vị trí video hợp lệ.


In [ ]:
# code
selected_id = (
    str(PILOT_ID_OVERRIDE) if PILOT_ID_OVERRIDE is not None
    else str(pilot.iloc[0]['id'])
)
selected_matches = pilot.loc[pilot['id'].eq(selected_id)]
if len(selected_matches) != 1:
    raise ValueError('ID được chọn phải xuất hiện đúng một lần trong pilot chung.')
selected_clip = selected_matches.iloc[0].to_dict()

archive_path = Path(str(selected_clip['source_archive']))
if not archive_path.is_absolute():
    archive_path = DATA_ROOT / archive_path
if not archive_path.is_file():
    raise FileNotFoundError(
        f'Không tìm thấy ZIP: {archive_path}\n'
        'Kiểm tra Raw.zip trên Drive và DATA_ROOT tại Task 2.2.'
    )

member_name = str(selected_clip['video_relpath'])
member_path = PurePosixPath(member_name)
if member_path.is_absolute() or '..' in member_path.parts or '\\' in member_name:
    raise ValueError('video_relpath phải là đường dẫn tương đối an toàn bên trong ZIP.')

with zipfile.ZipFile(archive_path, 'r') as archive:
    matches = [item for item in archive.infolist() if item.filename == member_name]
    if len(matches) != 1 or matches[0].is_dir():
        raise ValueError('Video phải xuất hiện đúng một lần trong ZIP.')
    member_info = matches[0]
    if member_info.file_size <= 0:
        raise ValueError('Video có kích thước 0 byte.')
    if member_info.file_size != int(selected_clip['video_size_bytes']):
        raise ValueError('Kích thước video trong ZIP không khớp manifest.')

print('ID bắt đầu tích hợp:', selected_id)
print('Split:', selected_clip['split'])
print('ZIP:', archive_path)
print('Video bên trong ZIP:', member_name)
print('Kích thước:', member_info.file_size, 'byte')
print('Transcript tham khảo:', selected_clip['raw_text'])

selected_metadata_issues = metadata_issues.loc[metadata_issues['id'].eq(selected_id)]
if selected_metadata_issues.empty:
    print('ID này không có cảnh báo trong metadata_issues.csv.')
else:
    display(selected_metadata_issues)


## Task 2.5 — Trích video pilot và kiểm tra khả năng đọc
Trích đúng một video vào thư mục tạm, giữ nguyên nội dung và thời gian.
Dùng ffprobe để kiểm tra luồng hình ảnh, số frame đọc được, FPS và kích thước.
Xem video và ghi nhận: có thấy mặt người nói không, có che khuất hay nhiều khuôn mặt không.

Kết quả cần có: video mở được, thông số hợp lệ và báo cáo của clip.
Đọc được video chưa chứng minh OpenFace phát hiện được khuôn mặt.


In [ ]:
# code
video_path = (LOCAL_MEDIA_ROOT / member_name).resolve()
if not video_path.is_relative_to(LOCAL_MEDIA_ROOT.resolve()):
    raise ValueError('Đường dẫn giải nén nằm ngoài thư mục làm việc.')
video_path.parent.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(archive_path, 'r') as archive:
    info = archive.getinfo(member_name)
    if not video_path.is_file() or video_path.stat().st_size != info.file_size:
        temporary_path = video_path.with_name(video_path.name + '.part')
        with archive.open(info) as source, temporary_path.open('wb') as destination:
            shutil.copyfileobj(source, destination)
        temporary_path.replace(video_path)

ffprobe_bin = shutil.which('ffprobe')
if ffprobe_bin is None:
    raise RuntimeError('Chưa có ffprobe trong runtime. Cần cài FFmpeg rồi chạy lại cell này.')

probe = subprocess.run(
    [ffprobe_bin, '-v', 'error', '-select_streams', 'v:0', '-count_frames',
     '-show_entries',
     'stream=codec_name,width,height,avg_frame_rate,nb_read_frames,duration,start_time:format=duration',
     '-of', 'json', str(video_path)],
    capture_output=True, text=True, timeout=120,
)
if probe.returncode != 0:
    raise RuntimeError(f'ffprobe không đọc được video:\n{probe.stderr[:1500]}')
if probe.stderr.strip():
    raise RuntimeError(f'Có lỗi giải mã cần kiểm tra:\n{probe.stderr[:1500]}')
probe_data = json.loads(probe.stdout)
streams = probe_data.get('streams', [])
if not streams:
    raise ValueError('Không tìm thấy luồng video.')
stream = streams[0]

try:
    fps = float(Fraction(stream.get('avg_frame_rate', '0/1')))
except (ValueError, ZeroDivisionError):
    raise ValueError('Không xác định được FPS từ video.') from None
frames_read = pd.to_numeric(stream.get('nb_read_frames'), errors='coerce')
duration = pd.to_numeric(stream.get('duration'), errors='coerce')
if not np.isfinite(duration) or duration <= 0:
    duration = pd.to_numeric(probe_data.get('format', {}).get('duration'), errors='coerce')
width = int(stream.get('width', 0))
height = int(stream.get('height', 0))
if (
    not np.isfinite(fps) or fps <= 0
    or not np.isfinite(frames_read) or frames_read <= 0
    or not np.isfinite(duration) or duration <= 0
    or width <= 0 or height <= 0
):
    raise ValueError('Thông số video không hợp lệ. Cần kiểm tra clip.')

stream_start = pd.to_numeric(stream.get('start_time'), errors='coerce')

pilot_video_report = {
    'id': selected_id, 'split': selected_clip['split'],
    'video_relpath': member_name, 'codec': stream.get('codec_name'),
    'duration_sec': float(duration), 'fps': fps,
    'stream_start_time_sec': float(stream_start) if np.isfinite(stream_start) else None,
    'decoded_frames': int(frames_read), 'width': width, 'height': height,
    'media_decode_status': 'ok', 'openface_extraction_status': 'not_run',
}
# Mỗi ID có báo cáo riêng; đường dẫn local chỉ dùng trong phiên Colab hiện tại.
pilot_report_path = REPORT_DIR / f'{selected_id}_video_check.json'
pilot_report_path.write_text(
    json.dumps(pilot_video_report, ensure_ascii=False, indent=2), encoding='utf-8'
)
display(pd.Series(pilot_video_report, name='Giá trị').to_frame())
display(Video(
    str(video_path), embed=True,
    mimetype=mimetypes.guess_type(str(video_path))[0],
    html_attributes='controls',
))
print('Quan sát: thấy mặt người nói không? Có che khuất hoặc nhiều khuôn mặt không?')
print('Báo cáo đã lưu:', pilot_report_path)


## Task 2.6 — Kiểm tra môi trường OpenFace
Ghi nhận runtime, FFmpeg và vị trí chương trình `FeatureExtraction`.
Nếu chưa cài OpenFace, trạng thái `not_found` là kết quả kiểm tra môi trường.
Cell này chỉ kiểm tra, chưa cài OpenFace hoặc trích đặc trưng.

Đầu ra Visual cần bàn giao: `features` float32 dạng `(T_vision, D_vision)`,
`timestamps` theo giây, `feature_names`, `success`, `confidence` và
`observed_mask` boolean cùng kích thước với features.
Chốt các cột AU intensity, hướng đầu và ánh nhìn sau khi xem kết quả OpenFace trên pilot;
chưa tự đặt số chiều hoặc ngưỡng confidence.

Kết quả cần có: báo cáo môi trường để chuẩn bị cài và chạy OpenFace.
Bàn giao chuỗi theo frame, giữ vị trí thời gian của frame lỗi, chưa Z-score hay cắt/padding.

Nguồn: [OpenFace chính thức](https://github.com/TadasBaltrusaitis/OpenFace),
[Command line arguments](https://github.com/TadasBaltrusaitis/OpenFace/wiki/Command-line-arguments).


In [ ]:
# code
candidate_bins = []
if OPENFACE_BIN_OVERRIDE is not None:
    candidate_bins.append(Path(OPENFACE_BIN_OVERRIDE))
else:
    executable_on_path = shutil.which('FeatureExtraction')
    if executable_on_path:
        candidate_bins.append(Path(executable_on_path))
    candidate_bins.extend([
        OPENFACE_ROOT / 'build' / 'bin' / 'FeatureExtraction',
        OPENFACE_ROOT / 'bin' / 'FeatureExtraction',
    ])

OPENFACE_BIN = next((p.resolve() for p in candidate_bins if p.is_file()), None)
openface_status = 'not_found'
missing_libraries = []
ldd_note = ''
if OPENFACE_BIN is not None:
    if not os.access(OPENFACE_BIN, os.X_OK):
        openface_status = 'binary_not_executable'
    else:
        openface_status = 'binary_found_not_run'
        ldd_bin = shutil.which('ldd')
        if platform.system() == 'Linux' and ldd_bin:
            dependency_check = subprocess.run(
                [ldd_bin, str(OPENFACE_BIN)],
                capture_output=True, text=True, timeout=30,
            )
            dependency_text = dependency_check.stdout + dependency_check.stderr
            missing_libraries = [
                line.strip() for line in dependency_text.splitlines()
                if 'not found' in line
            ]
            ldd_note = dependency_text.strip()
            if missing_libraries:
                openface_status = 'missing_shared_libraries'

ffmpeg_bin = shutil.which('ffmpeg')
ffmpeg_version = None
if ffmpeg_bin:
    ffmpeg_check = subprocess.run(
        [ffmpeg_bin, '-version'], capture_output=True, text=True, timeout=30,
    )
    ffmpeg_version = ffmpeg_check.stdout.splitlines()[0] if ffmpeg_check.stdout else None

# Đây là thông tin môi trường, không phải kết quả trích đặc trưng.
environment_report = {
    'checked_at_utc': datetime.now(timezone.utc).isoformat(),
    'python': platform.python_version(), 'platform': platform.platform(),
    'numpy': np.__version__, 'pandas': pd.__version__,
    'ffmpeg_path': ffmpeg_bin, 'ffmpeg_version': ffmpeg_version,
    'ffprobe_path': shutil.which('ffprobe'),
    'selected_pilot_id': selected_id,
    'openface_binary': str(OPENFACE_BIN) if OPENFACE_BIN else None,
    'openface_status': openface_status,
    'missing_libraries': missing_libraries,
    'openface_extraction_completed': False,
}
environment_path = REPORT_DIR / 'visual_environment.json'
environment_path.write_text(
    json.dumps(environment_report, ensure_ascii=False, indent=2), encoding='utf-8'
)
if ldd_note:
    (REPORT_DIR / 'openface_dynamic_libraries.txt').write_text(ldd_note, encoding='utf-8')

display(pd.Series(environment_report, name='Giá trị').to_frame())
if OPENFACE_BIN is None:
    print('Chưa tìm thấy FeatureExtraction. Cần cài/build OpenFace trước khi trích đặc trưng.')
else:
    print('Đã tìm thấy binary; vẫn cần chạy thử để kiểm chứng model và khả năng trích xuất.')
print('Chưa chạy OpenFace và chưa có kết quả đặc trưng Visual.')
print('Báo cáo đã lưu:', environment_path)


## Task 2.7 — Kiểm tra công cụ trước khi cài OpenFace

Kiểm tra phiên bản Ubuntu và các công cụ biên dịch để lựa chọn
cách cài OpenFace phù hợp. Cell này chỉ kiểm tra, chưa cài đặt.

In [ ]:
print("=== HỆ ĐIỀU HÀNH ===")
print(Path("/etc/os-release").read_text())

checks = {
    "CMake": ["cmake", "--version"],
    "Trình biên dịch C++": ["g++", "--version"],
    "OpenCV C++": ["pkg-config", "--modversion", "opencv4"],
    "dlib C++": [
        "dpkg-query", "-W", "-f=${Version}", "libdlib-dev"
    ],
    "OpenBLAS": [
        "dpkg-query", "-W", "-f=${Version}", "libopenblas-dev"
    ],
}

for name, command in checks.items():
    if shutil.which(command[0]) is None:
        print(f"{name}: chưa có công cụ kiểm tra")
        continue

    result = subprocess.run(
        command, capture_output=True, text=True, timeout=30
    )

    if result.returncode == 0:
        lines = result.stdout.strip().splitlines()
        print(f"{name}: {lines[0] if lines else 'không có thông tin'}")
    else:
        print(f"{name}: chưa tìm thấy qua lệnh kiểm tra")

In [ ]:
# TASK 2.8 — CÀI CÁC THƯ VIỆN C++
packages = [
    "build-essential",
    "cmake",
    "git",
    "wget",
    "pkg-config",
    "libopencv-dev",
    "libdlib-dev",
    "libopenblas-dev",
    "libboost-filesystem-dev",
    "libboost-system-dev",
]

print("Đang cập nhật danh sách gói...")
subprocess.run(
    ["apt-get", "update", "-qq"],
    check=True,
)

print("Đang cài thư viện...")
subprocess.run(
    ["apt-get", "install", "-y", "--no-install-recommends", *packages],
    check=True,
    env={**os.environ, "DEBIAN_FRONTEND": "noninteractive"},
)

# Đọc phiên bản sau khi cài; lỗi sẽ được báo ngay.
version_commands = {
    "OpenCV C++": ["pkg-config", "--modversion", "opencv4"],
    "dlib C++": [
        "dpkg-query", "-W", "-f=${Version}", "libdlib-dev"
    ],
    "OpenBLAS": [
        "dpkg-query", "-W", "-f=${Version}", "libopenblas-dev"
    ],
}

print("\n=== PHIÊN BẢN ĐÃ CÀI ===")
for name, command in version_commands.items():
    version = subprocess.check_output(
        command, text=True
    ).strip()
    print(f"{name}: {version}")

print("\nĐã chuẩn bị thư viện. Chưa biên dịch OpenFace.")

## Task 2.9 — Tải mã nguồn và biên dịch OpenFace

Biên dịch là chuyển mã nguồn C++ thành chương trình FeatureExtraction
để sử dụng cho việc trích đặc trưng khuôn mặt.

Dùng tối đa 2 tiến trình biên dịch đồng thời để hạn chế tiêu thụ RAM.
Số 2 là cấu hình cài đặt, không liên quan đến số frame hoặc đặc trưng.

Kết quả cần có: file FeatureExtraction và mã phiên bản nguồn.
Chưa kết luận trích xuất thành công; cần kiểm tra model và chạy pilot.

In [ ]:
# code
BUILD_DIR = OPENFACE_ROOT / "build"
BUILD_LOG = REPORT_DIR / "openface_build.log"

# Lưu toàn bộ output để xem nguyên nhân nếu có lỗi.
def run_openface_setup(command):
    print("Đang chạy:", " ".join(command), flush=True)

    with BUILD_LOG.open("a", encoding="utf-8") as log:
        log.write("\nCOMMAND: " + " ".join(command) + "\n")
        log.flush()

        result = subprocess.run(
            command,
            stdout=log,
            stderr=subprocess.STDOUT,
        )

    if result.returncode != 0:
        lines = BUILD_LOG.read_text(
            encoding="utf-8", errors="replace"
        ).splitlines()
        print("\n=== 50 DÒNG CUỐI LOG ===")
        print("\n".join(lines[-50:]))
        raise RuntimeError(
            f"Lệnh thất bại. Log đầy đủ: {BUILD_LOG}"
        )

# Chỉ tải khi chưa có thư mục; không ghi đè thư mục có sẵn.
if not OPENFACE_ROOT.exists():
    run_openface_setup([
        "git", "clone", "--depth", "1",
        "https://github.com/TadasBaltrusaitis/OpenFace.git",
        str(OPENFACE_ROOT),
    ])
elif not (OPENFACE_ROOT / ".git").is_dir():
    raise RuntimeError(
        f"{OPENFACE_ROOT} đã tồn tại nhưng không phải Git repository."
    )

# Ghi lại chính xác phiên bản mã nguồn đang sử dụng.
openface_commit = subprocess.check_output(
    ["git", "-C", str(OPENFACE_ROOT), "rev-parse", "HEAD"],
    text=True,
).strip()

(REPORT_DIR / "openface_source_commit.txt").write_text(
    openface_commit + "\n", encoding="utf-8"
)

# CMake xác định thư viện và tạo cấu hình biên dịch.
run_openface_setup([
    "cmake",
    "-S", str(OPENFACE_ROOT),
    "-B", str(BUILD_DIR),
    "-DCMAKE_BUILD_TYPE=Release",
])

# Biên dịch chương trình trích đặc trưng và các thư viện cần thiết.
run_openface_setup([
    "cmake", "--build", str(BUILD_DIR),
    "--target", "FeatureExtraction",
    "--parallel", "2",
])

OPENFACE_BIN = BUILD_DIR / "bin" / "FeatureExtraction"
if not OPENFACE_BIN.is_file() or not os.access(OPENFACE_BIN, os.X_OK):
    raise RuntimeError("Chưa tạo được FeatureExtraction có quyền thực thi.")

print("\nĐÃ BIÊN DỊCH XONG")
print("Chương trình:", OPENFACE_BIN)
print("Mã phiên bản nguồn:", openface_commit)
print("Log:", BUILD_LOG)
print("Chưa kiểm tra model và chưa chạy trích đặc trưng.")

In [ ]:
# code LƯU OPENFACE TRÊN DRIVE
binary_path = OPENFACE_ROOT / "build" / "bin" / "FeatureExtraction"

if not binary_path.is_file():
    raise RuntimeError(
        "Cần chạy thành công Task 2.9 trước khi lưu bản biên dịch."
    )

OPENFACE_CACHE_DIR = VISUAL_ROOT / "tools"
OPENFACE_CACHE_DIR.mkdir(parents=True, exist_ok=True)

print("Đang nén và lưu OpenFace lên Drive...", flush=True)

backup_path = Path(shutil.make_archive(
    base_name=str(OPENFACE_CACHE_DIR / "OpenFace_colab"),
    format="gztar",
    root_dir=str(OPENFACE_ROOT.parent),
    base_dir=OPENFACE_ROOT.name,
))

print("Đã lưu:", backup_path)
print("Dung lượng:", round(backup_path.stat().st_size / 1024**2, 1), "MiB")

## Task 2.11 — Bổ sung model cho OpenFace

Tải các model CE-CLM đã huấn luyện của tác giả để OpenFace
xác định các điểm đặc trưng trên khuôn mặt.

Bốn file tương ứng với bốn mức scale trong cấu hình CE-CLM,
không phải số lượng frame được lấy mẫu.

Kiểm tra sơ bộ file tải về và đặt model cạnh chương trình đã
biên dịch. Việc model hoạt động được sẽ kiểm chứng khi chạy pilot.

In [ ]:
# code
if not OPENFACE_BIN.is_file():
    raise RuntimeError("Cần biên dịch thành công Task 2.9 trước.")

model_source_dir = (
    OPENFACE_ROOT / "lib/local/LandmarkDetector/model/patch_experts"
)
model_runtime_dir = OPENFACE_BIN.parent / "model" / "patch_experts"

model_source_dir.mkdir(parents=True, exist_ok=True)
model_runtime_dir.mkdir(parents=True, exist_ok=True)

# Các địa chỉ từ download_models.sh chính thức của OpenFace.
model_urls = {
    "cen_patches_0.25_of.dat":
        "https://www.dropbox.com/s/7na5qsjzz8yfoer/cen_patches_0.25_of.dat?dl=1",
    "cen_patches_0.35_of.dat":
        "https://www.dropbox.com/s/k7bj804cyiu474t/cen_patches_0.35_of.dat?dl=1",
    "cen_patches_0.50_of.dat":
        "https://www.dropbox.com/s/ixt4vkbmxgab1iu/cen_patches_0.50_of.dat?dl=1",
    "cen_patches_1.00_of.dat":
        "https://www.dropbox.com/s/2t5t1sdpshzfhpj/cen_patches_1.00_of.dat?dl=1",
}

model_rows = []

for filename, url in model_urls.items():
    source_path = model_source_dir / filename
    needs_download = (
        not source_path.is_file()
        or source_path.stat().st_size == 0
    )

    candidate_path = source_path

    if needs_download:
        print("Đang tải:", filename, flush=True)
        candidate_path = source_path.with_suffix(".dat.part")

        subprocess.run(
            [
                "wget",
                "--timeout=60", "--tries=2", "--no-verbose",
                "-O", str(candidate_path), url,
            ],
            check=True,
        )

    if candidate_path.stat().st_size == 0:
        raise RuntimeError(f"File model rỗng: {filename}")

    # Phát hiện trường hợp tải nhầm trang web thay vì file model.
    with candidate_path.open("rb") as file:
        header = file.read(512).lstrip().lower()

    if header.startswith((b"<!doctype html", b"<html", b"<?xml")):
        raise RuntimeError(
            f"{filename}: tải về trang web hoặc thông báo lỗi."
        )

    if needs_download:
        candidate_path.replace(source_path)

    # Đặt model vào nơi chương trình đã biên dịch sẽ đọc.
    shutil.copy2(source_path, model_runtime_dir / filename)

    model_rows.append({
        "model": filename,
        "dung_luong_MiB": round(
            source_path.stat().st_size / 1024**2, 2
        ),
    })

display(pd.DataFrame(model_rows))
print("Đã chuẩn bị 4 file model. Chưa chạy trích đặc trưng.")

## Task 2.12 — Trích đặc trưng Visual trên một clip pilot

Chạy OpenFace trên clip đã chọn từ pilot chung, xử lý toàn bộ frame.
Yêu cầu xuất Action Units, hướng đầu và ánh nhìn để khảo sát đầu ra.

Giữ timestamp, confidence, success và cả các dòng có success = 0.
Chưa loại frame, điền thiếu, chuẩn hóa hoặc chốt số chiều đặc trưng.

Lưu CSV gốc, lệnh chạy, log và bảng tóm tắt trên Google Drive.
Mỗi lần chạy có thư mục riêng để tránh nhầm với kết quả cũ.

Kết quả cần có: CSV có dữ liệu, các nhóm đặc trưng được xuất
và số frame được OpenFace báo xử lý thành công.

In [ ]:
# TASK 2.12 — CHẠY OPENFACE TRÊN MỘT CLIP PILOT
if OPENFACE_BIN is None or not OPENFACE_BIN.is_file():
    raise RuntimeError("Chưa tìm thấy chương trình FeatureExtraction.")

if not video_path.is_file():
    raise FileNotFoundError("Video tạm đã mất. Chạy lại Task 2.4 và 2.5.")

landmark_model = (
    OPENFACE_BIN.parent / "model" / "main_ceclm_general.txt"
)
if not landmark_model.is_file():
    raise FileNotFoundError(f"Thiếu cấu hình model: {landmark_model}")

# Mỗi lần chạy lưu vào một thư mục riêng trên Drive.
started_at = datetime.now(timezone.utc)
run_name = started_at.strftime("%Y%m%dT%H%M%S_%fZ")
PILOT_OUTPUT_DIR = VISUAL_ROOT / "pilot" / selected_id / run_name
PILOT_OUTPUT_DIR.mkdir(parents=True, exist_ok=False)

OPENFACE_CSV_PATH = PILOT_OUTPUT_DIR / "features.csv"
OPENFACE_LOG_PATH = PILOT_OUTPUT_DIR / "openface.log"

command = [
    str(OPENFACE_BIN.resolve()),
    "-f", str(video_path.resolve()),
    "-mloc", str(landmark_model.resolve()),
    "-out_dir", str(PILOT_OUTPUT_DIR.resolve()),
    "-of", "features",
    "-aus",
    "-pose",
    "-gaze",
]

(PILOT_OUTPUT_DIR / "command.json").write_text(
    json.dumps(
        {"id": selected_id, "command": command},
        ensure_ascii=False, indent=2,
    ),
    encoding="utf-8",
)

print("Đang xử lý ID:", selected_id, flush=True)
print("Log được lưu tại:", OPENFACE_LOG_PATH, flush=True)

with OPENFACE_LOG_PATH.open("w", encoding="utf-8") as log:
    result = subprocess.run(
        command,
        cwd=str(OPENFACE_BIN.parent),
        stdout=log,
        stderr=subprocess.STDOUT,
    )

if result.returncode != 0 or not OPENFACE_CSV_PATH.is_file():
    log_lines = OPENFACE_LOG_PATH.read_text(
        encoding="utf-8", errors="replace"
    ).splitlines()
    print("\n".join(log_lines[-30:]))
    raise RuntimeError("OpenFace chưa tạo kết quả hợp lệ. Xem log phía trên.")

# Đọc CSV gốc, chỉ bỏ khoảng trắng quanh tên cột trong bộ nhớ.
visual_raw = pd.read_csv(OPENFACE_CSV_PATH, skipinitialspace=True)
visual_raw.columns = visual_raw.columns.str.strip()

required_columns = ["frame", "timestamp", "confidence", "success"]
missing_columns = [
    column for column in required_columns
    if column not in visual_raw.columns
]
if visual_raw.empty or missing_columns:
    raise ValueError(
        f"CSV rỗng hoặc thiếu cột bắt buộc: {missing_columns}"
    )

au_columns = [
    column for column in visual_raw.columns
    if column.startswith("AU") and column.endswith("_r")
]
pose_columns = [
    column for column in visual_raw.columns
    if column.startswith("pose_")
]
gaze_columns = [
    column for column in visual_raw.columns
    if column.startswith("gaze_")
]

if not au_columns or not pose_columns or not gaze_columns:
    raise ValueError("Thiếu nhóm AU intensity, head pose hoặc gaze. Cần xem log.")

success_values = pd.to_numeric(visual_raw["success"], errors="coerce")
if not success_values.isin([0, 1]).all():
    raise ValueError("Cột success có giá trị ngoài quy ước 0/1.")

summary = {
    "id": selected_id,
    "so_dong_frame": len(visual_raw),
    "so_frame_success_1": int(success_values.eq(1).sum()),
    "so_frame_success_0": int(success_values.eq(0).sum()),
    "ty_le_success_phan_tram": float(success_values.eq(1).mean() * 100),
    "so_cot_CSV": len(visual_raw.columns),
    "so_cot_AU_intensity": len(au_columns),
    "so_cot_pose": len(pose_columns),
    "so_cot_gaze": len(gaze_columns),
    "thoi_gian_chay_giay": (
        datetime.now(timezone.utc) - started_at
    ).total_seconds(),
}

(PILOT_OUTPUT_DIR / "summary.json").write_text(
    json.dumps(summary, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

display(pd.Series(summary, name="Giá trị").to_frame())
display(visual_raw[required_columns].head(5))
print("CSV gốc đã lưu:", OPENFACE_CSV_PATH)

### Kết luận Task 2.12

OpenFace đã tạo CSV gồm 158 dòng theo frame cho clip pilot
7JsX8y1ysxY$_$12. Có 158/158 dòng được báo success = 1,
tương ứng 100%; không có dòng success = 0.

CSV có 334 cột, trong đó có 17 cột AU intensity, 6 cột hướng/vị trí
đầu và 8 cột ánh nhìn. Thời gian chạy thử khoảng 18,04 giây.
CSV gốc, lệnh chạy, log và bảng tóm tắt đã được lưu trên Drive.

Kết quả xác nhận OpenFace đã chạy và xuất dữ liệu trên một clip.
Chưa kết luận theo dõi đúng người nói hoặc các đặc trưng đều chính xác;
cần kiểm tra trực quan và đối chiếu số frame với video gốc.

In [ ]:
# TASK 2.13 — ĐƯA NOTEBOOK VISUAL LÊN NHÁNH RIÊNG
import base64
import hashlib
import json
import requests
from getpass import getpass
from google.colab import _message, userdata

REPO = "danghau2512/DataMining-MSA"
BRANCH = "feat/preprocess-visual"
NOTEBOOK_PATH = "src/notebooks/02_preprocessing_Visual_BaoHoang.ipynb"

def push_visual_notebook():
    # Lấy notebook đang mở, gồm cả các cell mới thêm trên Colab.
    reply = _message.blocking_request("get_ipynb", timeout_sec=60)
    notebook = reply.get("ipynb") if isinstance(reply, dict) else None
    if isinstance(notebook, str):
        notebook = json.loads(notebook)
    if (not isinstance(notebook, dict)
            or notebook.get("nbformat") != 4 or not notebook.get("cells")):
        raise RuntimeError("Không lấy được notebook. Chạy cell trong giao diện Colab.")

    # Chỉ làm sạch bản xuất; không xóa output trên giao diện đang mở.
    for cell in notebook["cells"]:
        if cell.get("cell_type") == "code":
            cell["outputs"] = []
            cell["execution_count"] = None
            cell.setdefault("metadata", {}).pop("executionInfo", None)
            cell["metadata"].pop("outputId", None)
    notebook.setdefault("metadata", {}).pop("widgets", None)
    notebook["metadata"].setdefault("colab", {})["name"] = NOTEBOOK_PATH.split("/")[-1]
    notebook_bytes = (
        json.dumps(notebook, ensure_ascii=False, indent=2) + "\n"
    ).encode("utf-8")

    print("Repo:", REPO)
    print("Nhánh:", BRANCH)
    print("File:", NOTEBOOK_PATH)
    print("Số cell:", len(notebook["cells"]))

    # Nếu Colab không đọc được Secret, chuyển sang ô nhập token ẩn.
    try:
        token = userdata.get("token_datamining")
    except (
        userdata.SecretNotFoundError,
        userdata.NotebookAccessError,
        userdata.TimeoutException,
    ):
        print("Colab chưa đọc được Secret. Nhập token vào ô ẩn bên dưới.")
        token = getpass("Nhập GitHub token có quyền ghi repo (ô ẩn): ")
    token = token.strip()
    if not token:
        raise ValueError("Chưa có GitHub token.")

    with requests.Session() as session:
        session.headers.update({
            "Authorization": "Bearer " + token,
            "Accept": "application/vnd.github+json",
            "X-GitHub-Api-Version": "2026-03-10",
        })
        root = "https://api.github.com/repos/" + REPO

        def api(method, route="", allow_missing=False, **kwargs):
            response = session.request(
                method, root + ("/" + route if route else ""),
                timeout=60, **kwargs,
            )
            if response.status_code == 404 and allow_missing:
                return None
            if not response.ok:
                try:
                    message = response.json().get("message", "GitHub từ chối yêu cầu.")
                except ValueError:
                    message = "Phản hồi không phải JSON."
                raise RuntimeError(
                    f"GitHub HTTP {response.status_code}: {message}. "
                    "Kiểm tra thông báo; với HTTP 401/403, kiểm tra token và quyền ghi repo."
                )
            return response.json()

        try:
            repo_info = api("GET")
            if repo_info.get("permissions", {}).get("push") is False:
                raise RuntimeError("Tài khoản của token chưa có quyền ghi repository.")
            base_branch = repo_info["default_branch"]
            if BRANCH == base_branch:
                raise ValueError("Cần dùng nhánh riêng cho phần Visual.")

            # Nhánh mới bắt đầu từ nhánh mặc định; nhánh đã có thì dùng tiếp.
            branch_info = api(
                "GET", "git/ref/heads/" + BRANCH, allow_missing=True,
            )
            if branch_info is None:
                base = api("GET", "git/ref/heads/" + base_branch)
                api("POST", "git/refs", json={
                    "ref": "refs/heads/" + BRANCH,
                    "sha": base["object"]["sha"],
                })
                print("Đã tạo nhánh:", BRANCH)

            # Chỉ tạo/cập nhật đúng một file notebook Visual.
            route = "contents/" + NOTEBOOK_PATH
            previous = api(
                "GET", route, params={"ref": BRANCH}, allow_missing=True,
            )
            blob_sha = hashlib.sha1(
                b"blob " + str(len(notebook_bytes)).encode("ascii")
                + b"\0" + notebook_bytes
            ).hexdigest()

            if previous and previous.get("sha") == blob_sha:
                print("GitHub đã có đúng phiên bản này.")
            else:
                body = {
                    "message": "Add/update Bao Hoang visual preprocessing notebook",
                    "branch": BRANCH,
                    "content": base64.b64encode(notebook_bytes).decode("ascii"),
                }
                if previous:
                    body["sha"] = previous["sha"]
                result = api("PUT", route, json=body)
                print("Đã đưa notebook lên GitHub.")
                print("Commit:", result["commit"]["html_url"])

            print("Notebook:", f"https://github.com/{REPO}/blob/{BRANCH}/{NOTEBOOK_PATH}")
        finally:
            session.headers.pop("Authorization", None)
            token = None

push_visual_notebook()
